In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
!uv pip install tensorflow

Using Python 3.12.10 environment at: /home/anon-labs/Documents/projects/AlphaGoSimplified/.venv
Audited 1 package in 3ms


In [2]:
import tensorflow as tf
model=tf.keras.models.load_model('/home/anon-labs/projects/AlphaGoSimplified/files/value_conn.h5')

2025-10-07 23:14:47.801341: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-07 23:14:47.841042: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-07 23:14:48.853342: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/anon-labs/Documents/projects/AlphaGoSimplified/.venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)

* **7.1 Rerun the second code cell in Section 7.1.2 to play against the evaluation agent.
Try your best to win and see if you can beat the evaluation agent.**

In [3]:
from copy import deepcopy
from utils.ch07util import position_eval

def eval_move(env,model):
    # create a dictionary to hold all values
    values={}
    # iteratre through all possible next moves
    for m in env.validinputs:
        # make a hypothetical move
        env_copy=deepcopy(env)
        s,r,d,_=env_copy.step(m)
        # evaluate the hypothetical game state
        value=position_eval(env_copy,model)
        # add value to the dictionary
        if env.turn=="red":
            values[m]=round(value,5)
        # multiply value by -1 for yellow
        else:
            values[m]=round(-value,5)
    # choose the move with the highest evaluation    
    action = max(values,key=values.get)        
    return action, values

In [8]:
from utils.conn_simple_env import conn

env=conn()
state=env.reset()  
print(f"the current state is \n{state.T[::-1]}") 
while True:
    action, values=eval_move(env,model)
    print(f"evaluations of future moves are\n{values}")   
    print(f"the red player chose column {action}")
    state, reward, done, info=env.step(action)
    if done: 
        print(f"the current state is \n{state.T[::-1]}")
        print("the red player won")
        break    
    # the opponent chooses random moves   
    action=int(input("what's your move?"))
    print(f"the yellow player chose column {action}")
    state, reward, done, info=env.step(action)
    print(f"the current state is \n{state.T[::-1]}")
    if done: 
        if reward==-1:
            print("the yellow player won")
        else:
            print("game over, it's a tie")
        break

the current state is 
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]]
evaluations of future moves are
{1: np.float32(-0.49165), 2: np.float32(0.4506), 3: np.float32(0.39475), 4: np.float32(-0.33056), 5: np.float32(0.07255), 6: np.float32(0.13224), 7: np.float32(-0.31471)}
the red player chose column 2
the yellow player chose column 1
the current state is 
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [-1  1  0  0  0  0  0]]
evaluations of future moves are
{1: np.float32(0.38002), 2: np.float32(0.88281), 3: np.float32(0.99157), 4: np.float32(0.67626), 5: np.float32(0.80291), 6: np.float32(0.66971), 7: np.float32(0.28246)}
the red player chose column 3
the yellow player chose column 1
the current state is 
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [-1  0  0  0  0  0  0]
 [-1  1  1  0  0  0  0]]
evaluations

* **7.2 Modify the two code cells in Section 7.3.2 so that the depth is 2 in both functions
MiniMax_conn() and MiniMax_conn_eval(). See how often the evaluation-
augmented MiniMax agent wins.**

In [9]:
from utils.ch06util import MiniMax_conn
from utils.ch07util import MiniMax_conn_eval

results=[]
for i in range(10):
    state=env.reset() 
    if i%2==0:
        action=MiniMax_conn(env,depth=2)    
        state,reward,done,_=env.step(action)
    while True:
        action=MiniMax_conn_eval(env,model,depth=2) 
        state,reward,done,_=env.step(action)
        if done: 
            results.append(abs(reward))
            break 
        action=MiniMax_conn(env,depth=3) 
        state,reward,done,_=env.step(action)
        if done: 
            results.append(-abs(reward))
            break

In [10]:
# count how many times MiniMax with evaluation won
wins=results.count(1)
print(f"MiniMax with evaluation won {wins} games")
# count how many times MiniMax with evaluation lost
losses=results.count(-1)
print(f"MiniMax with evaluation lost {losses} games")
# count tie games
ties=results.count(0)
print(f"the game is tied {ties} times")          

MiniMax with evaluation won 2 games
MiniMax with evaluation lost 8 games
the game is tied 0 times
